# Ноутбук создан для 4 пункта чеклиста (создание и исследование классических моделей)

Препроцессинг и CV-харнесс вынесены в preprocessing.py и validation.py (общие для этого и simple_baseline_and_experimentations ноутбуков), чтобы не дублировать код. Загрузим данные и применим зафиксированный препроцессинг

In [6]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from preprocessing import preprocess_data_advanced
from validation import cross_validate_model, evaluate_model, train_kfold_and_predict, tune_hyperparameters

In [7]:
train_data = pd.read_csv("data/train.csv")
test_data = pd.read_csv("data/test.csv")
test_passenger_ids = test_data['PassengerId']

train_data, artifacts = preprocess_data_advanced(train_data, is_train=True)
test_data = preprocess_data_advanced(test_data, is_train=False, artifacts=artifacts)

categorical_cols = ['Pclass', 'Embarked', 'Title']
train_data = pd.get_dummies(train_data, columns=categorical_cols)
test_data = pd.get_dummies(test_data, columns=categorical_cols)
test_data = test_data.reindex(columns=train_data.drop('Survived', axis=1).columns, fill_value=0)

scale_cols = ['Age', 'Fare', 'SibSp', 'Parch', 'TicketGroupSize']

scaler = StandardScaler()
train_data[scale_cols] = scaler.fit_transform(train_data[scale_cols])
test_data[scale_cols] = scaler.transform(test_data[scale_cols])

X_train = train_data.drop(['Survived'], axis=1)
y_train = train_data['Survived']
X_test = test_data

X_train.shape, X_test.shape

((891, 18), (418, 18))

Заведём словарь для результатов всех моделей, пригодится для финального сравнения (пункт чеклиста "сложить все результаты и сравнить в конце"). `evaluate_model` из validation.py прогоняет модель через CV, печатает mean/std и сохраняет scores в results по ключу-названию

In [8]:
results = {}

Пункт 1: классическая логрег vs регуляризация (Lasso/Ridge/ElasticNet). В sklearn 1.9 параметр penalty устарел, вместо него используются l1_ratio (0 = чистый Ridge/L2, 1 = чистый Lasso/L1, между ними = ElasticNet) и C. C=inf означает отсутствие регуляризации. Перебор через tune_hyperparameters

In [9]:
from functools import partial
from sklearn.linear_model import LogisticRegression

# partial фиксирует solver/max_iter/random_state, чтобы не передавать их в каждой комбинации
# tune_hyperparameters вызовет logreg_ctor(**params), где params - только то, что перебираем (C, l1_ratio, class_weight)
logreg_ctor = partial(LogisticRegression, solver='saga', max_iter=5000, random_state=42)

# список из 2 словарей: tune_hyperparameters переберет все комбинации внутри каждого словаря отдельно
logreg_param_grid = [
    {'C': [np.inf], 'class_weight': [None, 'balanced']},  # без регуляризации (baseline)
    {'l1_ratio': [0, 0.25, 0.5, 0.75, 1], 'C': [0.01, 0.1, 1, 10, 100], 'class_weight': [None, 'balanced']},  # 0=Ridge, 1=Lasso, между ними ElasticNet
]

best_params, best_score = tune_hyperparameters(
    'logreg_regularization', logreg_ctor, logreg_param_grid, X_train, y_train, results
)

logreg_regularization: лучший скор 0.83387 +- 0.01288, параметры {'C': 1, 'class_weight': None, 'l1_ratio': 0.75}


Сравним solver='saga' с дефолтным 'lbfgs'. lbfgs поддерживает только L2-регуляризацию или её отсутствие (l1_ratio=0 или C=inf), l1/elasticnet ему sklearn не даёт, поэтому сетка для него уже, чем для saga

In [10]:
lbfgs_ctor = partial(LogisticRegression, solver='lbfgs', max_iter=5000, random_state=42)

lbfgs_param_grid = [
    {'C': [np.inf], 'class_weight': [None, 'balanced']},
    {'l1_ratio': [0], 'C': [0.01, 0.1, 1, 10, 100], 'class_weight': [None, 'balanced']},
]

best_params_lbfgs, best_score_lbfgs = tune_hyperparameters(
    'logreg_solver_lbfgs', lbfgs_ctor, lbfgs_param_grid, X_train, y_train, results
)

logreg_solver_lbfgs: лучший скор 0.83386 +- 0.01599, параметры {'C': 1, 'class_weight': None, 'l1_ratio': 0}


In [11]:
results

{'logreg_regularization': array([0.8547486 , 0.82022472, 0.8258427 , 0.8258427 , 0.84269663]),
 'logreg_solver_lbfgs': array([0.8603352 , 0.81460674, 0.8258427 , 0.8258427 , 0.84269663])}

Перебрал C, l1_ratio, class_weight и solver, лучший скор 0.834, почти не отличается от дефолтных параметров. Регуляризация и балансировка классов тут не нужны

Пункт 2: KNN, перебор n_neighbors/weights/metric

In [12]:
from sklearn.neighbors import KNeighborsClassifier

knn_param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 15, 21, 31],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'chebyshev'],
}

best_params_knn, best_scores_knn = tune_hyperparameters(
    'knn', KNeighborsClassifier, knn_param_grid, X_train, y_train, results
)

knn: лучший скор 0.82604 +- 0.02219, параметры {'metric': 'manhattan', 'n_neighbors': 5, 'weights': 'uniform'}


Лучший результат 0.826 на manhattan, n_neighbors=5, uniform, хуже логрега. Расстояния между one-hot категориями менее осмысленны, чем между числовыми фичами, поэтому KNN проигрывает

Пункт 3: Decision Tree, перебор criterion/max_depth/min_samples_split/min_samples_leaf/max_features/class_weight

In [13]:
from sklearn.tree import DecisionTreeClassifier

tree_ctor = partial(DecisionTreeClassifier, random_state=42)

tree_param_grid = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'max_features': [None, 'sqrt'],
    'class_weight': [None, 'balanced'],
}

best_params_tree, best_scores_tree = tune_hyperparameters(
    'decision_tree', tree_ctor, tree_param_grid, X_train, y_train, results
)

decision_tree: лучший скор 0.82714 +- 0.02302, параметры {'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': None, 'min_samples_leaf': 5, 'min_samples_split': 2}


Лучший скор 0.827 на min_samples_leaf=5, max_depth не ограничена (регуляризация только через размер листа). Примерно на уровне KNN, хуже логрега. Одно дерево нестабильно, ждём, что Random Forest как ансамбль деревьев даст прирост

Пункт 4: Random Forest, перебор n_estimators/max_depth/min_samples_leaf/max_features/criterion/class_weight. n_jobs=-1 фиксирован (использует все ядра для ускорения, на результат не влияет)

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf_ctor = partial(RandomForestClassifier, random_state=42, n_jobs=-1)

rf_param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [5, 10, None],
    'min_samples_leaf': [1, 5],
    'max_features': ['sqrt', None],
    'criterion': ['gini', 'entropy'],
    'class_weight': [None, 'balanced'],
}

best_params_rf, best_scores_rf = tune_hyperparameters(
    'random_forest', rf_ctor, rf_param_grid, X_train, y_train, results
)

random_forest: лучший скор 0.83948 +- 0.01287, параметры {'class_weight': None, 'criterion': 'entropy', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 300}


Лучший скор 0.839 на n_estimators=300, max_depth=10, criterion=entropy, max_features=sqrt. Впервые обогнали логрег (0.834), усреднение по 300 деревьям снизило разброс одного дерева (std упал с 0.023 до 0.013)

Пункт 5: бустинги. По чеклисту нужно минимум 2 из CatBoost/LightGBM/XGBoost, берём XGBoost и LightGBM (оба уже поставлены через pip, requirements.txt появится позже на этапе финального пайплайна). Перебор n_estimators/max_depth/learning_rate/subsample/colsample_bytree

In [15]:
from xgboost import XGBClassifier

xgb_ctor = partial(XGBClassifier, random_state=42)

xgb_param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 1.0],
    'colsample_bytree': [0.7, 1.0],
}

best_params_xgb, best_scores_xgb = tune_hyperparameters(
    'xgboost', xgb_ctor, xgb_param_grid, X_train, y_train, results
)

xgboost: лучший скор 0.84622 +- 0.02192, параметры {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300, 'subsample': 1.0}


Лучший скор 0.846 на max_depth=3 (неглубокие деревья), learning_rate=0.05, n_estimators=300, colsample_bytree=0.7. Новый лидер, обогнали Random Forest (0.839). Небольшая глубина + много деревьев с низким learning_rate лучше, чем несколько глубоких

In [16]:
from lightgbm import LGBMClassifier

# verbosity=-1 глушит служебные логи LightGBM (по умолчанию печатает инфо про сплиты на каждой итерации)
lgbm_ctor = partial(LGBMClassifier, random_state=42, verbosity=-1)

lgbm_param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [3, 5, -1],  # -1 = без ограничения глубины
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 1.0],
    'colsample_bytree': [0.7, 1.0],
}

best_params_lgbm, best_scores_lgbm = tune_hyperparameters(
    'lightgbm', lgbm_ctor, lgbm_param_grid, X_train, y_train, results
)

lightgbm: лучший скор 0.84510 +- 0.01845, параметры {'colsample_bytree': 0.7, 'learning_rate': 0.01, 'max_depth': -1, 'n_estimators': 300, 'subsample': 0.7}


Лучший скор 0.845 на max_depth без ограничения, learning_rate=0.01 (низкий), n_estimators=300, subsample=0.7. Почти как XGBoost (0.846), чуть ниже, но разброс меньше (std 0.018 против 0.022). Оба бустинга сейчас лидируют среди всех моделей

In [17]:
results

{'logreg_regularization': array([0.8547486 , 0.82022472, 0.8258427 , 0.8258427 , 0.84269663]),
 'logreg_solver_lbfgs': array([0.8603352 , 0.81460674, 0.8258427 , 0.8258427 , 0.84269663]),
 'knn': array([0.82681564, 0.83707865, 0.78651685, 0.85393258, 0.8258427 ]),
 'decision_tree': array([0.84916201, 0.84831461, 0.78651685, 0.83146067, 0.82022472]),
 'random_forest': array([0.8603352 , 0.84269663, 0.82022472, 0.83707865, 0.83707865]),
 'xgboost': array([0.86592179, 0.87078652, 0.82022472, 0.85393258, 0.82022472]),
 'lightgbm': array([0.8603352 , 0.87078652, 0.82022472, 0.84269663, 0.83146067])}

Пункт 5 продолжение: CatBoost на текущей one-hot предобработке (вариант без one-hot через cat_features пробовали, но CatBoost на CPU с Ordered boosting оказался слишком медленным для полного перебора сетки, убрали)

In [18]:
from catboost import CatBoostClassifier

# verbose=False глушит лог по итерациям, allow_writing_files=False отключает catboost_info/ в рабочей папке
# thread_count=-1 явно (использует все ядра, обычно и так дефолт)
catboost_ctor = partial(CatBoostClassifier, random_state=42, verbose=False, allow_writing_files=False, thread_count=-1)

catboost_param_grid = {
    'iterations': [100, 300],
    'depth': [4, 6, 8],
    'learning_rate': [0.03, 0.1],
    'l2_leaf_reg': [1, 5],
}

best_params_cb_onehot, best_scores_cb_onehot = tune_hyperparameters(
    'catboost_onehot', catboost_ctor, catboost_param_grid, X_train, y_train, results
)

catboost_onehot: лучший скор 0.84398 +- 0.00581, параметры {'depth': 4, 'iterations': 300, 'l2_leaf_reg': 1, 'learning_rate': 0.03}


Лучший скор 0.844 на depth=4, iterations=300, learning_rate=0.03. Разброс минимальный среди всех моделей (std 0.006), но средний скор чуть ниже XGBoost (0.846) и LightGBM (0.845)

Сведём все результаты в одну таблицу

In [19]:
summary = pd.DataFrame({
    'mean': {name: scores.mean() for name, scores in results.items()},
    'std': {name: scores.std() for name, scores in results.items()},
}).sort_values('mean', ascending=False)

summary

,mean,std
xgboost,0.846218,0.021921
lightgbm,0.845101,0.018453
catboost_onehot,0.843983,0.005806
random_forest,0.839483,0.012865
logreg_regularization,0.833871,0.012876
logreg_solver_lbfgs,0.833865,0.015989
decision_tree,0.827136,0.023022
knn,0.826037,0.022193


Итог по классическим моделям и бустингам: лидируют все 3 бустинга (XGBoost 0.846, LightGBM 0.845, CatBoost 0.844), Random Forest близко позади (0.839), логрег держится середняком без регуляризации (0.834), KNN и Decision Tree внизу (0.826-0.827). Разброс между лидерами меньше 0.002, разница внутри погрешности, ни один бустинг не выигрывает уверенно у других. Baseline из простого logreg (0.799 без фичей и нормирования) поднялся до 0.846 полным циклом препроцессинга и подбора гиперпараметров, дальше основной потенциал роста, скорее всего, не в замене модели, а в feature engineering и ансамблях

# Feature Engineering раунд 2

Пункт чеклиста: "попробуйте комбинации фичей (например, наиболее высоких по feature importance)". Обучим лучший XGBoost на всех данных и посмотрим на feature_importances_, чтобы понять, какие фичи комбинировать

In [20]:
best_xgb = XGBClassifier(
    colsample_bytree=0.7, learning_rate=0.05, max_depth=3, n_estimators=300, subsample=1.0, random_state=42
)
best_xgb.fit(X_train, y_train)

feature_importances = pd.Series(best_xgb.feature_importances_, index=X_train.columns).sort_values(ascending=False)
feature_importances

Title_Mr           0.257457
Title_Miss         0.210443
Sex                0.141424
Pclass_3           0.098351
Title_Mrs          0.041660
HasCabin           0.031358
TicketGroupSize    0.026833
Pclass_2           0.024543
Title_Rare         0.023307
Title_Master       0.022939
SibSp              0.021558
Pclass_1           0.019483
Embarked_Q         0.018250
Embarked_S         0.017779
Fare               0.016590
Age                0.012511
Parch              0.008130
Embarked_C         0.007384
dtype: float32

Топ по важности: Title_Mr/Miss/Mrs (~51% суммарно), Sex, Pclass_3, HasCabin, TicketGroupSize. SibSp/Parch по отдельности слабые (0.022/0.008). Попробуем: Sex×Pclass (категориальное взаимодействие, "женщина 3 класса" вело себя иначе, чем просто женщина или просто 3 класс), IsAlone и IsChild (агрегации SibSp/Parch/Age, которые по отдельности слабы)

In [21]:
train_data_fe = pd.read_csv("data/train.csv")
test_data_fe = pd.read_csv("data/test.csv")

train_data_fe, artifacts_fe = preprocess_data_advanced(train_data_fe, is_train=True)
test_data_fe = preprocess_data_advanced(test_data_fe, is_train=False, artifacts=artifacts_fe)

for df in (train_data_fe, test_data_fe):
    df['IsAlone'] = ((df['SibSp'] + df['Parch']) == 0).astype(int)
    df['IsChild'] = (df['Age'] < 16).astype(int)
    df['Sex_Pclass'] = df['Sex'].astype(str) + '_' + df['Pclass'].astype(str)

categorical_cols_fe = ['Pclass', 'Embarked', 'Title', 'Sex_Pclass']
train_data_fe = pd.get_dummies(train_data_fe, columns=categorical_cols_fe)
test_data_fe = pd.get_dummies(test_data_fe, columns=categorical_cols_fe)
test_data_fe = test_data_fe.reindex(columns=train_data_fe.drop('Survived', axis=1).columns, fill_value=0)

scale_cols_fe = ['Age', 'Fare', 'SibSp', 'Parch', 'TicketGroupSize']
scaler_fe = StandardScaler()
train_data_fe[scale_cols_fe] = scaler_fe.fit_transform(train_data_fe[scale_cols_fe])
test_data_fe[scale_cols_fe] = scaler_fe.transform(test_data_fe[scale_cols_fe])

X_train_fe = train_data_fe.drop(['Survived'], axis=1)
y_train_fe = train_data_fe['Survived']
X_test_fe = test_data_fe

X_train_fe.shape

(891, 26)

In [22]:
xgb_fixed = XGBClassifier(
    colsample_bytree=0.7, learning_rate=0.05, max_depth=3, n_estimators=300, subsample=1.0, random_state=42
)
evaluate_model('xgboost_fe2', xgb_fixed, X_train_fe, y_train_fe, results)

xgboost_fe2: 0.84396 +- 0.02194


array([0.87709497, 0.85955056, 0.81460674, 0.83146067, 0.83707865])

Новые фичи не помогли: 0.844 против 0.846 у XGBoost без них. Логично: XGBoost - дерево, оно и так умеет находить взаимодействия вроде "женщина + 3 класс" через последовательные сплиты, явно прописанное взаимодействие Sex×Pclass для него избыточно, а не полезно (лишние колонки только размывают сплиты). Ручной feature engineering сильнее помогает линейным моделям, которые взаимодействия сами не находят, но там регуляризация и так уже давала прирост не через фичи. Похоже, при текущем наборе базовых фичей дальше выжимать из ручной генерации почти нечего

Проверим гипотезу, что новые фичи полезнее для линейной модели (сама взаимодействия искать не умеет, в отличие от дерева)

In [23]:
logreg_fixed = LogisticRegression(solver='saga', max_iter=5000, random_state=42, C=1, l1_ratio=0.75)
evaluate_model('logreg_fe2', logreg_fixed, X_train_fe, y_train_fe, results)

logreg_fe2: 0.83500 +- 0.00938


array([0.84916201, 0.83146067, 0.8258427 , 0.84269663, 0.8258427 ])

Гипотеза не подтвердилась: логрег с новыми фичами дал 0.835 против 0.834 без них - разница внутри шума (хотя std заметно упал, 0.009 против 0.013, модель стала стабильнее). И для дерева, и для линейной модели этот набор комбинаций почти ничего не даёт. Title/Sex/Pclass и так несут почти весь сигнал по отдельности, добавление их произведений избыточно. На этом наборе фичей feature engineering раунд 2 исчерпан, новые модели строить не будем - переходим к ансамблям